# ActivationLens | Databricks Product Analytics Lakehouse

**What this notebook does:** Generates **40,000 synthetic user signups** and **380,000 base product events**, injects known quality faults, and builds **Bronze**, **Silver**, and **Gold** managed Delta tables in Unity Catalog. It validates data quality, measures 7-day activation and retention, and produces SQL-ready experiment metrics.

**Run location:** Databricks notebook, attached to **Serverless** compute. Run **All** cells from the top. The data is synthetic, not Meta or a real customer's data.

**Output tables:** `bronze_signups`, `bronze_events`, `silver_events`, `silver_event_quarantine`, `gold_user_activation`, `gold_experiment_summary`, `gold_daily_engagement`, `gold_retention_d7`, `dq_run_metrics`.

This notebook is repeatable: rerunning rebuilds only those ActivationLens-named tables; it does not delete unrelated tables.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

# Use the active workspace catalog so Free Edition works without extra account settings.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "activationlens"
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
except Exception as exc:
    print("Cannot create activationlens schema. Falling back to default:", str(exc)[:200])
    SCHEMA = "default"

PREFIX = f"`{CATALOG}`.`{SCHEMA}`"
def table(name):
    return f"{PREFIX}.`{name}`"
def save_delta(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table(name))
    print("Wrote", table(name))

print("Catalog:", CATALOG, "Schema:", SCHEMA)

## 1. Generate deterministic synthetic data

The control and treatment groups contain 20,000 signups each. Synthetic activation is intentionally set to 21.1% (control) and 25.8% (treatment), making this a **pipeline demonstration**, not evidence of a real experiment effect. Raw source events include 5,627 duplicate event IDs and 7,127 invalid rows so quality checks can be verified exactly.

In [ ]:
users = (
    spark.range(40000).withColumnRenamed("id", "user_num")
    .withColumn("user_id", F.concat(F.lit("user_"), F.col("user_num").cast("string")))
    .withColumn("signup_date", F.date_add(F.lit("2026-09-01").cast("date"), (F.col("user_num") % 20).cast("int")))
    .withColumn("variant", F.when((F.col("user_num") % 2) == 0, F.lit("control")).otherwise(F.lit("treatment")))
    .withColumn("activated_expected", F.when((F.col("user_num") % 2) == 0,
        (F.floor(F.col("user_num") / 2) % 1000) < 211).otherwise(
        (F.floor(F.col("user_num") / 2) % 1000) < 258))
    .select("user_num", "user_id", "signup_date", "variant", "activated_expected")
)

raw = (
    spark.range(380000).withColumnRenamed("id", "event_num")
    .withColumn("user_num", (F.col("event_num") % 40000).cast("long"))
    .withColumn("slot", F.floor(F.col("event_num") / 40000).cast("int"))
    .join(users, "user_num", "inner")
    .withColumn("event_id", F.concat(F.lit("event_"), F.col("event_num").cast("string")))
    .withColumn("event_date", F.date_add(F.col("signup_date"), F.col("slot")))
    .withColumn("event_type", F.when(F.col("slot") == 0, F.lit("signed_up"))
        .when((F.col("slot") == 1) & F.col("activated_expected"), F.lit("job_created"))
        .when((F.col("slot") == 7) & ((F.col("user_num") % 4) < 2), F.lit("session_started"))
        .when(F.col("slot") == 7, F.lit("email_delivered"))
        .when((F.col("slot") % 3) == 0, F.lit("session_started"))
        .otherwise(F.lit("feature_viewed")))
    .withColumn("event_ts", F.to_timestamp(F.concat(F.col("event_date").cast("string"), F.lit(" 12:00:00"))))
    .withColumn("ingest_source", F.lit("base"))
    .select("event_id", "user_id", "event_type", "event_ts", "ingest_source")
)

# Controlled data quality faults: duplicate IDs and invalid missing user IDs.
duplicates = (raw.withColumn("n", F.regexp_extract(F.col("event_id"), r"(\d+)", 1).cast("long"))
    .where(F.col("n") < 5627).drop("n"))
duplicates = duplicates.withColumn("ingest_source", F.lit("duplicated"))
invalid = (spark.range(7127)
    .withColumn("event_id", F.concat(F.lit("invalid_"), F.col("id").cast("string")))
    .withColumn("user_id", F.lit(None).cast("string"))
    .withColumn("event_type", F.lit("session_started"))
    .withColumn("event_ts", F.to_timestamp(F.lit("2026-09-20 12:00:00")))
    .withColumn("ingest_source", F.lit("invalid"))
    .select(raw.columns))
bronze_events = raw.unionByName(duplicates).unionByName(invalid)
save_delta(users.drop("activated_expected"), "bronze_signups")
save_delta(bronze_events, "bronze_events")
print("Bronze event rows:", spark.table(table("bronze_events")).count(), "(expected 392,754)")

## 2. Validate, quarantine, and deduplicate

Quarantine invalid rows separately so records are not silently lost. Use an event ID as the deduplication key and prefer original ingested records over injected duplicates. Preserve a quality audit row for inspection.

In [ ]:
source = spark.table(table("bronze_events"))
allowed = ["signed_up", "job_created", "session_started", "feature_viewed", "email_delivered"]
valid = (
    F.col("event_id").isNotNull() & F.col("user_id").isNotNull()
    & F.col("event_type").isin(allowed) & F.col("event_ts").isNotNull()
)
quarantine = source.filter(~valid)
valid_events = source.filter(valid)
priority = Window.partitionBy("event_id").orderBy(F.when(F.col("ingest_source") == "base", 0).otherwise(1))
silver = (valid_events.withColumn("rn", F.row_number().over(priority))
    .filter(F.col("rn") == 1).drop("rn"))
save_delta(quarantine, "silver_event_quarantine")
save_delta(silver, "silver_events")

raw_count = source.count()
invalid_count = quarantine.count()
valid_count = valid_events.count()
clean_count = silver.count()
duplicate_count = valid_count - clean_count
assert raw_count == 392754, f"Unexpected raw row count {raw_count}"
assert invalid_count == 7127, f"Unexpected invalid count {invalid_count}"
assert duplicate_count == 5627, f"Unexpected duplicate count {duplicate_count}"
assert clean_count == 380000, f"Unexpected cleaned count {clean_count}"

quality = spark.createDataFrame([(
    datetime.now(timezone.utc).replace(tzinfo=None), raw_count, invalid_count, duplicate_count, clean_count
)], "run_utc timestamp, bronze_rows long, quarantined_rows long, duplicate_rows_removed long, silver_rows long")
save_delta(quality, "dq_run_metrics")
display(quality)

## 3. Build Gold analytics tables

The Gold layer answers product questions rather than exposing raw event logs: who activated within 7 days, cohort retention on day 7, daily active users, and the experiment breakdown. These are **simulated** results.

In [ ]:
users_src = spark.table(table("bronze_signups"))
events = spark.table(table("silver_events"))
job_events = (events.filter(F.col("event_type") == "job_created")
    .groupBy("user_id").agg(F.min(F.to_date("event_ts")).alias("first_job_date")))
gold_activation = (users_src.join(job_events, "user_id", "left")
    .withColumn("is_activated_7d", F.col("first_job_date").isNotNull()
        & (F.datediff(F.col("first_job_date"), F.col("signup_date")) >= 0)
        & (F.datediff(F.col("first_job_date"), F.col("signup_date")) <= 7))
    .select("user_id", "signup_date", "variant", "first_job_date", "is_activated_7d"))
save_delta(gold_activation, "gold_user_activation")

experiment = (gold_activation.groupBy("variant")
    .agg(F.count("*").alias("signups"), F.sum(F.col("is_activated_7d").cast("long")).alias("activated_7d"))
    .withColumn("activation_rate", F.round(F.col("activated_7d") / F.col("signups"), 4)))
save_delta(experiment, "gold_experiment_summary")

daily = (events.filter(F.col("event_type").isin("signed_up", "job_created", "session_started", "feature_viewed"))
    .withColumn("activity_date", F.to_date("event_ts"))
    .groupBy("activity_date")
    .agg(F.countDistinct("user_id").alias("daily_active_users"), F.count("*").alias("event_count")))
save_delta(daily, "gold_daily_engagement")

day7 = (events.filter(F.col("event_type") == "session_started")
    .join(users_src.select("user_id", "signup_date", "variant"), "user_id")
    .filter(F.datediff(F.to_date("event_ts"), F.col("signup_date")) == 7)
    .select("user_id").distinct().withColumn("returned_day_7", F.lit(True)))
retention = (users_src.join(day7, "user_id", "left")
    .withColumn("returned_day_7", F.coalesce("returned_day_7", F.lit(False)))
    .groupBy("signup_date", "variant")
    .agg(F.count("*").alias("cohort_size"), F.sum(F.col("returned_day_7").cast("long")).alias("retained_day_7"))
    .withColumn("retention_rate_d7", F.round(F.col("retained_day_7") / F.col("cohort_size"), 4)))
save_delta(retention, "gold_retention_d7")
display(experiment.orderBy("variant"))

## 4. Final validation and useful SQL

Expected experiment result: control **21.1%**, treatment **25.8%**. These percentages were deliberately built into synthetic data. The objective is to demonstrate a reliable end-to-end pipeline, not to claim a real-world uplift.

In [ ]:
results = {row["variant"]: row.asDict() for row in spark.table(table("gold_experiment_summary")).collect()}
assert results["control"]["signups"] == 20000
assert results["treatment"]["signups"] == 20000
assert results["control"]["activated_7d"] == 4220
assert results["treatment"]["activated_7d"] == 5160
assert abs(results["control"]["activation_rate"] - 0.211) < 0.00001
assert abs(results["treatment"]["activation_rate"] - 0.258) < 0.00001
print("PASS: 40K users, 380K clean events, 7,127 quarantined invalid rows, 5,627 duplicates removed")
print("PASS: SQL product activation rates, daily engagement, and D7 retention tables")
print("Tables available for SQL / Genie:")
for name in ["bronze_signups", "bronze_events", "silver_events", "silver_event_quarantine",
             "gold_user_activation", "gold_experiment_summary", "gold_daily_engagement", "gold_retention_d7", "dq_run_metrics"]:
    print(" ", f"{CATALOG}.{SCHEMA}.{name}")
print("Next: Catalog >", CATALOG, ">", SCHEMA, "> gold_experiment_summary")

## Next actions in Databricks UI

1. **Catalog** > select the catalog printed above > `activationlens` (or `default` if creation permissions required fallback). Confirm all nine tables exist.
2. **SQL Editor**: query `SELECT * FROM <catalog>.<schema>.gold_experiment_summary;` using the exact catalog and schema printed above.
3. **Genie Agents** > create a Genie space for ActivationLens and add `gold_experiment_summary`, `gold_daily_engagement`, `gold_retention_d7`, and `dq_run_metrics` as trusted datasets. Then send the Genie link in ChatGPT so the connected Databricks Genie integration can query it.
4. **Jobs & Pipelines** > create a job using this notebook as its task. Schedule only if your Databricks edition allows it. Reruns replace only the nine project tables.
5. In GitHub, publish this notebook and a README explaining the synthetic setup, table modeling, Bronze/Silver/Gold architecture, and validation assertions. Avoid calling this a production deployment.